In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

pd.set_option('display.max_columns', None)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from DATA.TOOLS.fetchPlayersStats import FetchPlayersStats
from DATA.TOOLS.fetchTeamStats import *

# feature_path = os.path.abspath(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor")
feature_path = os.path.abspath('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor')
# Add the parent directory to sys.path
if feature_path not in sys.path:
    sys.path.append(feature_path)

from FEATURE_ENGINEERING.features import *
from DATA.TOOLS.playerPositions import *

Features I plan on adding in the future

Player-specific usage and scoring data

- /BoxScoreScoringV2: Breaks down how a player scores (paint, midrange, 3s, free throws). *

Shot quality and location data

- /ShotChartDetail: Individual shot attempts, zones, and frequencies.

- /LeagueDashPlayerShotLocations: Aggregated shot location tendencies.

- /LeagueDashPlayerPtShot: Breaks down shooting by play type and situation.

Opponent and matchup data

- /LeagueDashPtDefend: How defenders contest shots and limit scoring. *

- /LeagueDashTeamStats with defense filters: Opponent’s defensive efficiency.

- /BoxScoreMatchupsV3: Player-vs-player defensive assignments.

Game context and pace

- /ScoreboardV2 or /PlayByPlayV2: For back-to-backs, rest, or pace indicators.

- /TeamDashboardByGeneralSplits: Team-level pace, offensive rating, and context.


In [2]:
pd.set_option('display.max_columns', None)
def convert_min_to_float(min_str):
    try:
        if isinstance(min_str, str) and ":" in min_str:
            minutes, seconds = map(int, min_str.split(":"))
            total_minutes = minutes + seconds / 60
            return round(total_minutes, 2)
        elif isinstance(min_str, (int, float)):
            return float(min_str)
        else:
            return 0
    except:
        return 0
    

s19_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S19.csv')
s19_regular['IS_PLAYOFF'] = 0
s19_regular['MIN'] = s19_regular['MIN'].apply(convert_min_to_float)

s20_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S20.csv')
s20_regular['IS_PLAYOFF'] = 0
s20_regular['MIN'] = s20_regular['MIN'].apply(convert_min_to_float)

s21_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S21.csv')
s21_regular['IS_PLAYOFF'] = 0
s21_regular['MIN'] = s21_regular['MIN'].apply(convert_min_to_float)

s22_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S22.csv')
s22_regular['IS_PLAYOFF'] = 0
s22_regular['MIN'] = s22_regular['MIN'].apply(convert_min_to_float)

s23_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S23.csv')
s23_regular['IS_PLAYOFF'] = 0
s23_regular['MIN'] = s23_regular['MIN'].apply(convert_min_to_float)

s24_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S24.csv')
s24_regular['IS_PLAYOFF'] = 0
s24_regular['MIN'] = s24_regular['MIN'].apply(convert_min_to_float)

s25_regular = pd.read_csv('../DATA/CSV_FILES/REGULAR_DATA/S25.csv')
s25_regular['IS_PLAYOFF'] = 0
s25_regular['MIN'] = s25_regular['MIN'].apply(convert_min_to_float)

### Boxscorematchupv3 from nba_api

In [5]:
import os
import pickle
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.exceptions import ReadTimeout, ConnectionError
from nba_api.stats.endpoints import boxscorematchupsv3

class MatchupDataFetcher:
    def __init__(self, cache_dir='matchup_cache', max_workers=3, sleep_time=1.5):
        self.cache_dir = cache_dir
        self.max_workers = max_workers
        self.sleep_time = sleep_time
        self.failed_games = []
        self.completed_games = set()
        
        # Create cache directory if it doesn't exist
        os.makedirs(cache_dir, exist_ok=True)
        
        # Load existing cache
        self.load_cache_status()
    
    def get_cache_file(self, game_id):
        """Get cache file path for a specific game"""
        return os.path.join(self.cache_dir, f'game_{game_id}.pkl')
    
    def is_cached(self, game_id):
        """Check if a game is already cached"""
        return os.path.exists(self.get_cache_file(game_id))
    
    def save_game_data(self, game_id, data):
        """Save game data to cache"""
        cache_file = self.get_cache_file(game_id)
        with open(cache_file, 'wb') as f:
            pickle.dump(data, f)
        self.completed_games.add(game_id)
    
    def load_game_data(self, game_id):
        """Load game data from cache"""
        cache_file = self.get_cache_file(game_id)
        if os.path.exists(cache_file):
            with open(cache_file, 'rb') as f:
                return pickle.load(f)
        return None
    
    def load_cache_status(self):
        """Load which games are already completed"""
        cache_files = [f for f in os.listdir(self.cache_dir) if f.startswith('game_') and f.endswith('.pkl')]
        self.completed_games = set(f.replace('game_', '').replace('.pkl', '') for f in cache_files)
        print(f"Found {len(self.completed_games)} cached games")
    
    def fetch_single_game(self, game_id, retry_count=3):
        """Fetch data for a single game with retry logic"""
        for attempt in range(retry_count):
            try:
                time.sleep(self.sleep_time)
                
                boxscore = boxscorematchupsv3.BoxScoreMatchupsV3(
                    game_id=f'00{game_id}',
                    timeout=60
                )
                data_frames = boxscore.get_data_frames()
                
                if data_frames and len(data_frames) > 0 and not data_frames[0].empty:
                    # Process the data
                    boxscore_df = data_frames[0]
                    boxscore_df['matchupMinutes'] = round(boxscore_df['matchupMinutesSort'] / 60, 2)
                    
                    def_df = (
                        boxscore_df.groupby('personIdDef')
                        .agg({
                        'gameId': 'first',
                        'teamId': 'first',
                        'matchupFieldGoalsMade': 'sum',
                        'matchupFieldGoalsAttempted': 'sum',
                        'matchupThreePointersMade': 'sum',
                        'matchupThreePointersAttempted': 'sum',
                        'playerPoints': 'sum',
                        'matchupMinutes': 'sum',
                        'matchupTurnovers': 'sum',
                        'matchupBlocks': 'sum',
                        'shootingFouls': 'sum',
                        'matchupAssists': 'sum'
                        })
                        .reset_index()
                    )
                    
                    # Calculate percentages safely
                    def_df['DEF_FG_PCT_ALLOWED'] = def_df.apply(
                        lambda row: round(row['matchupFieldGoalsMade'] / row['matchupFieldGoalsAttempted'], 3) 
                        if row['matchupFieldGoalsAttempted'] > 0 else 0, axis=1
                    )
                    def_df['DEF_3PT_PCT_ALLOWED'] = def_df.apply(
                        lambda row: round(row['matchupThreePointersMade'] / row['matchupThreePointersAttempted'], 3) 
                        if row['matchupThreePointersAttempted'] > 0 else 0, axis=1
                    )
                    def_df['PTS_ALLOWED_PER_MIN'] = def_df.apply(
                        lambda row: round(row['playerPoints'] / row['matchupMinutes'], 2) 
                        if row['matchupMinutes'] > 0 else 0, axis=1
                    )
                    def_df['DEF_TOV_FORCED_PER_MIN'] = def_df.apply(
                    lambda row: round(row['matchupTurnovers'] / row['matchupMinutes'], 2) 
                    if row['matchupMinutes'] > 0 else 0, axis=1
                )
                    def_df['DEF_BLOCKS_PER_MIN'] = def_df.apply(
                        lambda row: round(row['matchupBlocks'] / row['matchupMinutes'], 2) 
                        if row['matchupMinutes'] > 0 else 0, axis=1
                    )
                    def_df['DEF_SHOOTING_FOULS_PER_MIN'] = def_df.apply(
                        lambda row: round(row['shootingFouls'] / row['matchupMinutes'], 2) 
                        if row['matchupMinutes'] > 0 else 0, axis=1
                    )
                    def_df['DEF_AST_ALLOWED_PER_MIN'] = def_df.apply(
                        lambda row: round(row['matchupAssists'] / row['matchupMinutes'], 2) 
                        if row['matchupMinutes'] > 0 else 0, axis=1
                    )
                    return def_df, None
                else:
                    return None, "No data available"
                    
            except ReadTimeout:
                if attempt < retry_count - 1:
                    time.sleep(2 ** attempt)  # Exponential backoff
                    continue
                return None, "Timeout after retries"
            except Exception as e:
                if attempt < retry_count - 1:
                    time.sleep(1)
                    continue
                return None, str(e)
        
        return None, "Max retries exceeded"
    
    def fetch_games_bulk(self, game_ids, resume=True):
        """Fetch games in bulk with caching and resume capability"""
        if resume:
            # Filter out already completed games
            remaining_games = [gid for gid in game_ids if str(gid) not in self.completed_games]
            print(f"Resuming: {len(remaining_games)} games remaining out of {len(game_ids)}")
        else:
            remaining_games = game_ids
            print(f"Fetching all {len(game_ids)} games (not resuming)")
        
        if not remaining_games:
            print("All games already cached!")
            return self.load_all_cached_data(game_ids)
        
        print(f"Processing {len(remaining_games)} games using ThreadPoolExecutor...")
        
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            # Submit tasks for remaining games
            future_to_game = {
                executor.submit(self.fetch_single_game, game_id): game_id 
                for game_id in remaining_games
            }
            
            # Process completed tasks
            for i, future in enumerate(as_completed(future_to_game), 1):
                game_id = future_to_game[future]
                
                try:
                    result, error = future.result()
                    
                    if result is not None:
                        # Save to cache
                        self.save_game_data(str(game_id), result)
                        print(f"✓ Game {game_id} completed and cached ({i}/{len(remaining_games)})")
                    else:
                        self.failed_games.append((game_id, error))
                        print(f"✗ Game {game_id} failed: {error} ({i}/{len(remaining_games)})")
                        
                except Exception as e:
                    self.failed_games.append((game_id, str(e)))
                    print(f"✗ Game {game_id} exception: {str(e)} ({i}/{len(remaining_games)})")
        
        print(f"\nCompleted!")
        print(f"Successfully processed: {len(remaining_games) - len(self.failed_games)} games")
        print(f"Failed games: {len(self.failed_games)}")
        
        return self.load_all_cached_data(game_ids)
    
    def load_all_cached_data(self, game_ids):
        """Load all cached data for the given game IDs"""
        all_data = []
        missing_games = []
        
        for game_id in game_ids:
            cached_data = self.load_game_data(str(game_id))
            if cached_data is not None:
                all_data.append(cached_data)
            else:
                missing_games.append(game_id)
        
        if missing_games:
            print(f"Warning: {len(missing_games)} games not found in cache")
        
        if all_data:
            combined_data = pd.concat(all_data, ignore_index=True)
            print(f"Loaded {len(all_data)} games, total shape: {combined_data.shape}")
            return combined_data
        else:
            print("No cached data found!")
            return pd.DataFrame()
    
    def retry_failed_games(self):
        """Retry games that previously failed"""
        if not self.failed_games:
            print("No failed games to retry")
            return pd.DataFrame()
        
        print(f"Retrying {len(self.failed_games)} failed games...")
        failed_game_ids = [game_id for game_id, _ in self.failed_games]
        self.failed_games = []  # Clear the failed list
        
        return self.fetch_games_bulk(failed_game_ids, resume=False)
    
    def get_cache_stats(self):
        """Get statistics about the cache"""
        total_cached = len(self.completed_games)
        failed_count = len(self.failed_games)
        
        print(f"Cache Statistics:")
        print(f"  Cached games: {total_cached}")
        print(f"  Failed games: {failed_count}")
        print(f"  Cache directory: {self.cache_dir}")
        
        return {
            'cached_games': total_cached,
            'failed_games': failed_count,
            'cache_dir': self.cache_dir
        }

# Usage example:
# Initialize the fetcher
fetcher = MatchupDataFetcher(cache_dir='matchup_cache', max_workers=5, sleep_time=1.5)

# Get all game IDs
gameIds = s19_regular['GAME_ID'].unique()

# Fetch all games (will resume from cache if interrupted)
gameLogsTotal = fetcher.fetch_games_bulk(gameIds, resume=True)

# # Check cache statistics
fetcher.get_cache_stats()

# If you want to retry failed games later
# gameLogsTotal = fetcher.retry_failed_games()

Found 7043 cached games
Resuming: 7 games remaining out of 1230
Processing 7 games using ThreadPoolExecutor...
✗ Game 21800418 failed: list index out of range (1/7)
✗ Game 21800218 failed: list index out of range (2/7)
✗ Game 21800429 failed: list index out of range (3/7)
✗ Game 21800665 failed: list index out of range (4/7)
✗ Game 21800014 failed: list index out of range (5/7)
✗ Game 21800678 failed: list index out of range (6/7)
✗ Game 21800911 failed: list index out of range (7/7)

Completed!
Successfully processed: 0 games
Failed games: 7
Loaded 1223 games, total shape: (25931, 20)
Cache Statistics:
  Cached games: 7043
  Failed games: 7
  Cache directory: matchup_cache


{'cached_games': 7043, 'failed_games': 7, 'cache_dir': 'matchup_cache'}

In [6]:
gameLogsTotal.rename(columns={'personIdDef': 'PLAYER_ID', 'teamId': 'TEAM_ID', 'gameId': 'GAME_ID'}, inplace=True)
gameLogsTotal

,PLAYER_ID,GAME_ID,TEAM_ID,matchupFieldGoalsMade,matchupFieldGoalsAttempted,matchupThreePointersMade,matchupThreePointersAttempted,playerPoints,matchupMinutes,matchupTurnovers,matchupBlocks,shootingFouls,matchupAssists,DEF_FG_PCT_ALLOWED,DEF_3PT_PCT_ALLOWED,PTS_ALLOWED_PER_MIN,DEF_TOV_FORCED_PER_MIN,DEF_BLOCKS_PER_MIN,DEF_SHOOTING_FOULS_PER_MIN,DEF_AST_ALLOWED_PER_MIN
0,101161,0021800001,1610612738,6,9,0,1,13,4.46,0,0,1,2,0.667,0.000,2.91,0.00,0.00,0.22,0.45
1,200755,0021800001,1610612738,3,8,2,3,9,13.44,2,0,1,3,0.375,0.667,0.67,0.15,0.00,0.07,0.22
2,201143,0021800001,1610612755,11,28,1,6,23,11.25,1,3,0,4,0.393,0.167,2.04,0.09,0.27,0.00,0.36
3,202330,0021800001,1610612755,0,8,0,2,0,10.02,4,0,0,4,0.000,0.000,0.00,0.40,0.00,0.00,0.40
4,202681,0021800001,1610612755,5,7,1,1,11,10.71,0,0,0,1,0.714,1.000,1.03,0.00,0.00,0.00,0.09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25926,1628656,0021801223,1610612765,5,6,2,2,12,6.33,0,0,0,3,0.833,1.000,1.90,0.00,0.00,0.00,0.47
25927,1628971,0021801223,1610612752,1,7,1,4,4,10.04,3,1,1,4,0.143,0.250,0.40,0.30,0.10,0.10,0.40
25928,1628995,0021801223,1610612765,5,8,4,6,14,11.21,0,0,0,2,0.625,0.667,1.25,0.00,0.00,0.00,0.18
25929,1629011,0021801223,1610612765,5,11,3,6,18,9.57,1,0,2,7,0.455,0.500,1.88,0.10,0.00,0.21,0.73


In [7]:
def mergeMatchupDATA(data, matchup_data):
    data = data.copy()
    matchup_data = matchup_data.copy()
    

    if 'GAME_ID' in matchup_data.columns:
        matchup_data['GAME_ID'] = matchup_data['GAME_ID'].astype(int)
    
    # Ensure main data PLAYER_ID is also string
    if 'PLAYER_ID' in data.columns:
        data['PLAYER_ID'] = data['PLAYER_ID']
    
    merge_columns = [
        'GAME_ID', 'PLAYER_ID',
        'matchupFieldGoalsMade', 'matchupFieldGoalsAttempted',
        'matchupThreePointersMade', 'matchupThreePointersAttempted',
        'playerPoints', 'matchupMinutes', 'matchupFieldGoalsPercentage',
        'matchupThreePointersPercentage', 'DEF_FG_PCT_ALLOWED',
        'DEF_3PT_PCT_ALLOWED', 'PTS_ALLOWED_PER_MIN',
        'DEF_TOV_FORCED_PER_MIN', 'DEF_BLOCKS_PER_MIN', 'DEF_SHOOTING_FOULS_PER_MIN', 'DEF_AST_ALLOWED_PER_MIN'
    ]
    
    available_columns = [col for col in merge_columns if col in matchup_data.columns]
    matchup_subset = matchup_data[available_columns].copy()
    
    merged_data = data.merge(
        matchup_subset,
        on=['GAME_ID', 'PLAYER_ID'],
        how='left',
        suffixes=('', '_matchup')
    )
    return merged_data

df = mergeMatchupDATA(s19_regular, gameLogsTotal)
df.head()

,Unnamed: 0.1,Unnamed: 0,PLAYER_NAME,PLAYER_ID,MATCHUP,TEAM_ABBREVIATION,TEAM_ID,OPP_ABBREVIATION,HOME_GAME,GAME_ID,GAME_DATE,WL,PTS,AST,REB,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,STL,BLK,TOV,PLUS_MINUS,FANTASY_PTS,POINT_PER_SHOT,EFG,START_POSITION,COMMENT,OFF_RATING,E_OFF_RATING,DEF_RATING,E_DEF_RATING,NET_RATING,OREB_PCT,DREB_PCT,REB_PCT,AST_PCT,EFG_PCT,AST_TOV,USG_PCT,TS_PCT,E_PACE,PACE,PIE,POSS,PACE_PER40,E_USG_PCT,MIN,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,PTS_OFF_TOV,PTS_2ND_CHANCE,PTS_FB,PTS_PAINT,OPP_PTS_OFF_TOV,OPP_PTS_2ND_CHANCE,OPP_PTS_FB,OPP_PTS_PAINT,BLKA,PF,PFD,IS_PLAYOFF,TEAM_NAME,TEAM_MIN,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_STL,TEAM_BLK,TEAM_TOV,TEAM_PF,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_PACE,GAME_PACE,OPP_PACE,OPP_TEAM_ID,TEAM_OFF_RATING,TEAM_DEF_RATING,OPP_DEF_RATING,OPP_OFF_RATING,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV,whos_favored,spread,total,team_is_favored,team_spread,percentageFieldGoalsAttempted2pt,percentageFieldGoalsAttempted3pt,percentagePoints2pt,percentagePointsMidrange2pt,percentagePoints3pt,percentagePointsFastBreak,percentagePointsFreeThrow,percentagePointsOffTurnovers,percentagePointsPaint,percentageAssisted2pt,percentageUnassisted2pt,percentageAssisted3pt,percentageUnassisted3pt,percentageAssistedFGM,percentageUnassistedFGM,GP,AGE,FREQ_FG3,FG3M_main,FG3A_main,FG3_PCT_main,NS_FG3_PCT,PLUS_MINUS_FG3,FREQ_FG2,FG2M,FG2A,FG2_PCT,NS_FG2_PCT,PLUS_MINUS_FG2,FREQ_LT_06,FGM_LT_06,FGA_LT_06,LT_06_PCT,NS_LT_06_PCT,PLUS_MINUS_LT_06,FREQ_LT_10,FGM_LT_10,FGA_LT_10,LT_10_PCT,NS_LT_10_PCT,PLUS_MINUS_LT_10,FREQ_GT_15,FGM_GT_15,FGA_GT_15,GT_15_PCT,NS_GT_15_PCT,PLUS_MINUS_GT_15,matchupFieldGoalsMade,matchupFieldGoalsAttempted,matchupThreePointersMade,matchupThreePointersAttempted,playerPoints,matchupMinutes,DEF_FG_PCT_ALLOWED,DEF_3PT_PCT_ALLOWED,PTS_ALLOWED_PER_MIN,DEF_TOV_FORCED_PER_MIN,DEF_BLOCKS_PER_MIN,DEF_SHOOTING_FOULS_PER_MIN,DEF_AST_ALLOWED_PER_MIN
0,0,0,Daniel Theis,1628464,BOS vs. PHI,BOS,1610612738,PHI,1,21800001,2018-10-16,W,0,0,1,0,0,NaN,0,0,NaN,0,0,NaN,0,1,0,0,0,3,1.2,0.000,NaN,NaN,NaN,111.1,111.1,70.0,70.0,41.1,0.000,0.200,0.111,0.000,0.000,0.00,0.000,0.000,110.32,110.32,0.000,9.0,91.94,0.000,4.13,4.23,0.32,2,3,4,3,0,0,3,0,0,0.00,0,0,0.000,1,1,1.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,1.0,0.0,0,Boston Celtics,240,42,97,0.433,11,37,0.297,10,14,0.714,12,43,55,21,7,5,15,20,105,18,106.6,106.6,106.6,1610612755,98.9,82.0,98.0,81.2,87.0,34.0,87.0,0.391,47.0,18.0,8.0,5.0,16.0,home,4.5,211.5,True,4.5,0.000,0.000,0.000,0.000,0.000,0.0,0.000,0.000,0.000,0.0,0.0,0.0,0.0,0.000,0.000,66.0,27.0,0.261,0.73,2.17,0.336,0.350,-0.014,0.739,3.14,6.14,0.511,0.524,-0.013,0.409,2.02,3.39,0.594,0.611,-0.017,0.535,2.45,4.44,0.553,0.572,-0.019,0.401,1.17,3.33,0.350,0.362,-0.012,1.0,4.0,0.0,1.0,4.0,1.70,0.250,0.000,2.35,0.00,0.0,0.59,0.00
1,1,1,Klay Thompson,202691,GSW vs. OKC,GSW,1610612744,OKC,1,21800002,2018-10-16,W,14,0,4,5,20,0.250,1,8,0.125,3,3,1.0,1,3,0,0,2,2,16.8,0.657,0.275000,G,NaN,106.7,106.7,102.6,102.6,4.0,0.026,0.067,0.048,0.000,0.275,0.00,0.250,0.328,103.59,103.59,-0.021,75.0,86.33,0.250,34.98,4.35,2.72,3,5,8,45,0,0,22,0,6,0.00,5,14,0.357,2,2,1.000,3.0,2.0,0.0,4.0,18.0,14.0,8.0,26.0,2.0,3.0,3.0,0,Golden State Warriors,240,42,95,0.442,7,26,0.269,17,18,0.944,17,41,58,28,7,7,21,29,108,8,106.6,106.6,106.6,1610612760,101.0,93.5,101.6,94.1,100.0,33.0,91.0,0.363,45.0,21.0,12.0,6.0,15.0,home,12.0,220.5,True,12.0,0.600,0.400,0.571,0.286,0.214,0.0,0.214,0.214,0.286,0.5,0.5,1.0,0.0,0.600,0.400,78.0,29.0,0.330,1.31,3.65,0.358,0.357,0.000,0.670,3.40,7.42,0.458,0.507,-0.049,0.353,2.24,3.91,0.574,0.594,-0.020,0.469,2.60,5.19,0.501,0.553,-0.052,0.441,1.72,4.88,0.352,0.368,-0.016,4.0,12.0,1.0,4.0,11.0,14.90,0.333,0.250,0.74,0.20,0.0,0.13,0.07
2,2,2,Draymond Green,203110,GSW vs. OKC,GSW,1610612744,OKC,1,21800002,2018-10-16

### Leaguedashptdefend from nba_api

In [34]:
from nba_api.stats.endpoints import leaguedashptdefend
categories = ['3 Pointers', '2 Pointers', 'Less Than 6Ft', 'Less Than 10Ft', 'Greater Than 15Ft']
data = []
for category in categories:
    league_df = leaguedashptdefend.LeagueDashPtDefend(
        per_mode_simple='PerGame',
        defense_category=category,
        season='2018-19',
        season_type_all_star='Regular Season'
    ).get_data_frames()[0]
    league_df[f'FREQ_{category}'] = league_df['FREQ']
    league_df[f'PLUS_MINUS_{category}'] = league_df['PLUSMINUS']
    data.append(league_df)

league_df = pd.concat(data)
player_stats = league_df.groupby([
    'CLOSE_DEF_PERSON_ID', 
    'PLAYER_NAME', 
    'PLAYER_LAST_TEAM_ID', 
    'PLAYER_LAST_TEAM_ABBREVIATION', 
    'PLAYER_POSITION', 
    'AGE', 
    'GP'
]).agg('max').reset_index()

player_stats = player_stats[['CLOSE_DEF_PERSON_ID', 'PLAYER_NAME', 'PLAYER_LAST_TEAM_ID', 'PLAYER_LAST_TEAM_ABBREVIATION', 'PLAYER_POSITION', 'AGE','GP', 'FREQ_3 Pointers', 'FG3M', 'FG3A', 'FG3_PCT', 'NS_FG3_PCT','PLUS_MINUS_3 Pointers', 'FREQ_2 Pointers', 'FG2M', 'FG2A', 'FG2_PCT', 'NS_FG2_PCT', 'PLUS_MINUS_2 Pointers', 'FREQ_Less Than 6Ft', 'FGM_LT_06', 'FGA_LT_06', 'LT_06_PCT', 'NS_LT_06_PCT', 'PLUS_MINUS_Less Than 6Ft', 'FREQ_Less Than 10Ft', 'FGM_LT_10', 'FGA_LT_10', 'LT_10_PCT', 'NS_LT_10_PCT', 'PLUS_MINUS_Less Than 10Ft', 'FREQ_Greater Than 15Ft', 'FGM_GT_15', 'FGA_GT_15', 'GT_15_PCT', 'NS_GT_15_PCT', 'PLUS_MINUS_Greater Than 15Ft']]
player_stats.rename(columns={'CLOSE_DEF_PERSON_ID': 'PLAYER_ID','PLAYER_LAST_TEAM_ID':'TEAM_ID','FREQ_3 Pointers': 'FREQ_FG3', 'FREQ_2 Pointers': 'FREQ_FG2', 'FREQ_Less Than 6Ft': 'FREQ_LT_06', 'FREQ_Less Than 10Ft': 'FREQ_LT_10', 'FREQ_Greater Than 15Ft': 'FREQ_GT_15', 'PLUS_MINUS_3 Pointers': 'PLUS_MINUS_FG3','PLUS_MINUS_2 Pointers': 'PLUS_MINUS_FG2','PLUS_MINUS_Less Than 6Ft': 'PLUS_MINUS_LT_06', 'PLUS_MINUS_Less Than 10Ft': 'PLUS_MINUS_LT_10','PLUS_MINUS_Greater Than 15Ft': 'PLUS_MINUS_GT_15' }, inplace=True)
player_stats



,PLAYER_ID,PLAYER_NAME,TEAM_ID,PLAYER_LAST_TEAM_ABBREVIATION,PLAYER_POSITION,AGE,GP,FREQ_FG3,FG3M,FG3A,FG3_PCT,NS_FG3_PCT,PLUS_MINUS_FG3,FREQ_FG2,FG2M,FG2A,FG2_PCT,NS_FG2_PCT,PLUS_MINUS_FG2,FREQ_LT_06,FGM_LT_06,FGA_LT_06,LT_06_PCT,NS_LT_06_PCT,PLUS_MINUS_LT_06,FREQ_LT_10,FGM_LT_10,FGA_LT_10,LT_10_PCT,NS_LT_10_PCT,PLUS_MINUS_LT_10,FREQ_GT_15,FGM_GT_15,FGA_GT_15,GT_15_PCT,NS_GT_15_PCT,PLUS_MINUS_GT_15
0,1713,Vince Carter,1610612737,ATL,F-G,42.0,76,0.454,1.01,2.80,0.362,0.349,0.013,0.546,1.76,3.37,0.523,0.520,0.003,0.343,1.24,2.12,0.584,0.607,-0.023,0.426,1.46,2.63,0.555,0.567,-0.012,0.527,1.17,3.25,0.360,0.357,0.003
1,1717,Dirk Nowitzki,1610612742,DAL,F,41.0,51,0.283,0.76,2.33,0.328,0.351,-0.023,0.717,3.29,5.92,0.556,0.525,0.032,0.390,2.04,3.22,0.634,0.617,0.017,0.456,2.29,3.76,0.609,0.577,0.032,0.444,1.41,3.67,0.385,0.364,0.021
2,2037,Jamal Crawford,1610612756,PHX,G,39.0,59,0.347,0.51,1.61,0.316,0.352,-0.036,0.653,1.92,3.03,0.631,0.512,0.119,0.383,1.29,1.78,0.724,0.605,0.118,0.474,1.47,2.20,0.669,0.563,0.106,0.449,0.80,2.08,0.382,0.368,0.014
3,2199,Tyson Chandler,1610612747,LAL,C,36.0,55,0.203,0.73,2.13,0.342,0.352,-0.010,0.797,4.18,8.35,0.501,0.533,-0.032,0.403,2.53,4.22,0.599,0.629,-0.030,0.493,2.93,5.16,0.567,0.587,-0.020,0.382,1.45,4.00,0.364,0.366,-0.003
4,2200,Pau Gasol,1610612749,MIL,C-F,38.0,29,0.161,0.34,1.10,0.313,0.344,-0.032,0.839,3.10,5.76,0.539,0.536,0.003,0.402,1.45,2.76,0.525,0.633,-0.108,0.518,2.03,3.55,0.573,0.589,-0.016,0.367,1.03,2.52,0.411,0.361,0.050
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
518,1629234,Drew Eubanks,1610612759,SAS,F,22.0,19,0.319,0.26,1.16,0.227,0.350,-0.122,0.681,1.47,2.47,0.596,0.510,0.086,0.333,0.84,1.21,0.696,0.619,0.077,0.362,0.89,1.32,0.680,0.573,0.107,0.536,0.58,1.95,0.297,0.357,-0.060
519,1629244,Cam Reynolds,1610612750,MIN,G,24.0,17,0.387,0.59,2.12,0.278,0.366,-0.088,0.613,2.29,3.35,0.684,0.527,0.158,0.409,1.59,2.24,0.711,0.622,0.089,0.473,1.76,2.59,0.682,0.579,0.103,0.462,0.94,2.53,0.372,0.371,0.001
520,1629312,Haywood Highsmith,1610612755,PHI,F,22.0,4,0.182,0.25,0.50,0.500,0.343,0.158,0.818,1.00,2.25,0.444,0.445,-0.001,0.545,0.50,1.50,0.333,0.472,-0.138,0.636,0.75,1.75,0.429,0.434,-0.005,0.273,0.25,0.75,0.333,0.357,-0.023
521,1629353,Isaac Humphries,1610612737,ATL,C,21.0,5,0.367,1.00,2.20,0.455,0.337,0.118,0.633,2.00,3.80,0.526,0.523,0.004,0.400,1.40,2.40,0.583,0.635,-0.051,0.467,1.60,2.80,0.571,0.590,-0.019,0.433,1.20,2.60,0.462,0.373,0.089


In [35]:
def mergeDATA(s19_data, main_data):
    s19_data = s19_data.copy()
    main_data = main_data.copy()
    
    if 'TEAM_ID' in main_data.columns:
        main_data['TEAM_ID'] = main_data['TEAM_ID'].astype(int)
    
    # Ensure main data PLAYER_ID is also string
    if 'PLAYER_ID' in main_data.columns:
        main_data['PLAYER_ID'] = main_data['PLAYER_ID'].astype(str)
    
    # Ensure s19_data PLAYER_ID is also string
    if 'PLAYER_ID' in s19_data.columns:
        s19_data['PLAYER_ID'] = s19_data['PLAYER_ID'].astype(str)
    
    merge_columns = [
        'TEAM_ID', 'PLAYER_ID', 'GP',
        'AGE', 'FREQ_FG3', 'FG3M', 'FG3A', 'FG3_PCT', 'NS_FG3_PCT', 'PLUS_MINUS_FG3',
        'FREQ_FG2', 'FG2M', 'FG2A', 'FG2_PCT', 'NS_FG2_PCT', 'PLUS_MINUS_FG2',
        'FREQ_LT_06', 'FGM_LT_06', 'FGA_LT_06', 'LT_06_PCT', 'NS_LT_06_PCT', 'PLUS_MINUS_LT_06',
        'FREQ_LT_10', 'FGM_LT_10', 'FGA_LT_10', 'LT_10_PCT', 'NS_LT_10_PCT', 'PLUS_MINUS_LT_10',
        'FREQ_GT_15', 'FGM_GT_15', 'FGA_GT_15', 'GT_15_PCT', 'NS_GT_15_PCT', 'PLUS_MINUS_GT_15'
    ]
    
    available_columns = [col for col in merge_columns if col in main_data.columns]
    main_subset = main_data[available_columns].copy()
    merged_data = s19_data.merge(
        main_subset,
        on=['TEAM_ID', 'PLAYER_ID'],
        how='left',
        suffixes=('', '_main')
    )
    return merged_data

df = mergeDATA(s19_regular, player_stats)
df

,Unnamed: 0,PLAYER_NAME,PLAYER_ID,MATCHUP,TEAM_ABBREVIATION,TEAM_ID,OPP_ABBREVIATION,HOME_GAME,GAME_ID,GAME_DATE,WL,PTS,AST,REB,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,STL,BLK,TOV,PLUS_MINUS,FANTASY_PTS,POINT_PER_SHOT,EFG,START_POSITION,COMMENT,OFF_RATING,E_OFF_RATING,DEF_RATING,E_DEF_RATING,NET_RATING,OREB_PCT,DREB_PCT,REB_PCT,AST_PCT,EFG_PCT,AST_TOV,USG_PCT,TS_PCT,E_PACE,PACE,PIE,POSS,PACE_PER40,E_USG_PCT,MIN,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,PTS_OFF_TOV,PTS_2ND_CHANCE,PTS_FB,PTS_PAINT,OPP_PTS_OFF_TOV,OPP_PTS_2ND_CHANCE,OPP_PTS_FB,OPP_PTS_PAINT,BLKA,PF,PFD,IS_PLAYOFF,TEAM_NAME,TEAM_MIN,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_STL,TEAM_BLK,TEAM_TOV,TEAM_PF,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_PACE,GAME_PACE,OPP_PACE,OPP_TEAM_ID,TEAM_OFF_RATING,TEAM_DEF_RATING,OPP_DEF_RATING,OPP_OFF_RATING,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV,whos_favored,spread,total,team_is_favored,team_spread,percentageFieldGoalsAttempted2pt,percentageFieldGoalsAttempted3pt,percentagePoints2pt,percentagePointsMidrange2pt,percentagePoints3pt,percentagePointsFastBreak,percentagePointsFreeThrow,percentagePointsOffTurnovers,percentagePointsPaint,percentageAssisted2pt,percentageUnassisted2pt,percentageAssisted3pt,percentageUnassisted3pt,percentageAssistedFGM,percentageUnassistedFGM,GP,AGE,FREQ_FG3,FG3M_main,FG3A_main,FG3_PCT_main,NS_FG3_PCT,PLUS_MINUS_FG3,FREQ_FG2,FG2M,FG2A,FG2_PCT,NS_FG2_PCT,PLUS_MINUS_FG2,FREQ_LT_06,FGM_LT_06,FGA_LT_06,LT_06_PCT,NS_LT_06_PCT,PLUS_MINUS_LT_06,FREQ_LT_10,FGM_LT_10,FGA_LT_10,LT_10_PCT,NS_LT_10_PCT,PLUS_MINUS_LT_10,FREQ_GT_15,FGM_GT_15,FGA_GT_15,GT_15_PCT,NS_GT_15_PCT,PLUS_MINUS_GT_15
0,0,Daniel Theis,1628464,BOS vs. PHI,BOS,1610612738,PHI,1,21800001,2018-10-16,W,0,0,1,0,0,NaN,0,0,NaN,0,0,NaN,0,1,0,0,0,3,1.2,0.000,NaN,NaN,NaN,111.1,111.1,70.0,70.0,41.1,0.000,0.200,0.111,0.000,0.000,0.00,0.000,0.000,110.32,110.32,0.000,9.0,91.94,0.000,4.13,4.23,0.32,2,3,4,3,0,0,3,0,0,0.00,0,0,0.000,1,1,1.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,1.0,0.0,0,Boston Celtics,240,42,97,0.433,11,37,0.297,10,14,0.714,12,43,55,21,7,5,15,20,105,18,106.6,106.6,106.6,1610612755,98.9,82.0,98.0,81.2,87.0,34.0,87.0,0.391,47.0,18.0,8.0,5.0,16.0,home,4.5,211.5,True,4.5,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.00,0.00,0.00,0.00,0.000,0.000,66.0,27.0,0.261,0.73,2.17,0.336,0.350,-0.014,0.739,3.14,6.14,0.511,0.524,-0.013,0.409,2.02,3.39,0.594,0.611,-0.017,0.535,2.45,4.44,0.553,0.572,-0.019,0.401,1.17,3.33,0.350,0.362,-0.012
1,1,Klay Thompson,202691,GSW vs. OKC,GSW,1610612744,OKC,1,21800002,2018-10-16,W,14,0,4,5,20,0.250,1,8,0.125,3,3,1.000,1,3,0,0,2,2,16.8,0.657,0.275000,G,NaN,106.7,106.7,102.6,102.6,4.0,0.026,0.067,0.048,0.000,0.275,0.00,0.250,0.328,103.59,103.59,-0.021,75.0,86.33,0.250,34.98,4.35,2.72,3,5,8,45,0,0,22,0,6,0.00,5,14,0.357,2,2,1.000,3.0,2.0,0.0,4.0,18.0,14.0,8.0,26.0,2.0,3.0,3.0,0,Golden State Warriors,240,42,95,0.442,7,26,0.269,17,18,0.944,17,41,58,28,7,7,21,29,108,8,106.6,106.6,106.6,1610612760,101.0,93.5,101.6,94.1,100.0,33.0,91.0,0.363,45.0,21.0,12.0,6.0,15.0,home,12.0,220.5,True,12.0,0.600,0.400,0.571,0.286,0.214,0.000,0.214,0.214,0.286,0.50,0.50,1.00,0.00,0.600,0.400,78.0,29.0,0.330,1.31,3.65,0.358,0.357,0.000,0.670,3.40,7.42,0.458,0.507,-0.049,0.353,2.24,3.91,0.574,0.594,-0.020,0.469,2.60,5.19,0.501,0.553,-0.052,0.441,1.72,4.88,0.352,0.368,-0.016
2,2,Draymond Green,203110,GSW vs. OKC,GSW,1610612744,OKC,1,21800002,2018-10-16,W,2,5,13,1,6,0.167,0,1,0.000,0,0,NaN,1,12,3,0,6,2,28.1,0.333,0.166667,F,NaN,100.0,100.0,94.8,94.8,5.2,0.033,0.279,0.178,0.185,0.167,0.83,0.143,0.167,111.39,111.39,0.071,75.0,92.82,0.143,32.75,4.05,2.37,6,20,22,79,1,1,62,1,4,0.25,0,2,0.000,2,3,0.667,0.0,0.0,0.0,2.0,14.0,10.0,8.0,20.0,1.0,3.0,1.0,0,Golden State Warriors,240,42,95,0.442,7,26,0.269,17,18,0.944,17,41,58,28,7,7,21,29,108,8,106.6,106.6,106.6,16106127

### Boxscoringv3

In [ ]:
from nba_api.stats.endpoints import boxscorescoringv3
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import random
from requests.exceptions import ReadTimeout, ConnectionError

def fetch_game_data(gameId, sleep_time=1.0):
    """Fetch data for a single game with error handling"""
    try:
        time.sleep(sleep_time)
        
        boxscore = boxscorescoringv3.BoxScoreScoringV3(
            game_id=f'00{gameId}',
            timeout=60
        )
        data_frames = boxscore.get_data_frames()
        if data_frames and len(data_frames) > 0 and not data_frames[0].empty:
            return data_frames[0], gameId, None
        else:
            return None, gameId, "No data available"
            
    except ReadTimeout:
        return None, gameId, "Timeout"
    except Exception as e:
        return None, gameId, str(e)

# Get unique game IDs
gameIds = s25_regular['GAME_ID'].unique()
games = []
failed_games = []

print(f"Processing {len(gameIds)} games using ThreadPoolExecutor...")

max_workers = 3 
sleep_time = 1.5  

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    # Submit all tasks
    future_to_game = {
        executor.submit(fetch_game_data, gameId, sleep_time): gameId 
        for gameId in gameIds
    }
    
    # Process completed tasks
    for i, future in enumerate(as_completed(future_to_game), 1):
        gameId = future_to_game[future]
        
        try:
            result, game_id, error = future.result()
            
            if result is not None:
                games.append(result)
                print(f"✓ Game {gameId} completed ({i}/{len(gameIds)})")
            else:
                failed_games.append((gameId, error))
                print(f"✗ Game {gameId} failed: {error} ({i}/{len(gameIds)})")
                
        except Exception as e:
            failed_games.append((gameId, str(e)))
            print(f"✗ Game {gameId} exception: {str(e)} ({i}/{len(gameIds)})")

print(f"\nCompleted!")
print(f"Successfully processed: {len(games)} games")
print(f"Failed games: {len(failed_games)}")

if failed_games:
    print(f"Failed games: {failed_games[:10]}...")

if games:
    games_df = pd.concat(games, ignore_index=True)
    print(f"\nFinal dataset shape: {games_df.shape}")
    games_df.head()
else:
    print("No games were successfully processed!")

In [ ]:
def mergeDATA(s19_data, boxscoringv3_data):
    s19_data = s19_data.copy()
    boxscoringv3_data = boxscoringv3_data.copy()
    
    if 'gameId' in boxscoringv3_data.columns:
        boxscoringv3_data['GAME_ID'] = boxscoringv3_data['gameId'].astype(int)
    
    if 'personId' in boxscoringv3_data.columns:
        boxscoringv3_data['PLAYER_ID'] = boxscoringv3_data['personId']
    
    merge_columns = [
        'GAME_ID', 'PLAYER_ID', 
        'percentageFieldGoalsAttempted2pt', 'percentageFieldGoalsAttempted3pt', 
        'percentagePoints2pt', 'percentagePointsMidrange2pt', 'percentagePoints3pt',
        'percentagePointsFastBreak', 'percentagePointsFreeThrow', 'percentagePointsOffTurnovers', 
        'percentagePointsPaint', 'percentageAssisted2pt', 'percentageUnassisted2pt',
        'percentageAssisted3pt', 'percentageUnassisted3pt', 'percentageAssistedFGM', 
        'percentageUnassistedFGM'
    ]
    
    available_columns = [col for col in merge_columns if col in boxscoringv3_data.columns]
    boxscoringv3_subset = boxscoringv3_data[available_columns].copy()
    merged_data = s19_data.merge(
        boxscoringv3_subset,
        on=['GAME_ID', 'PLAYER_ID'],
        how='left',
        suffixes=('', '_boxscore')
    )
    return merged_data

df = mergeDATA(s25_regular, games_df)
df.drop(columns=['Unnamed: 0', 'TEAM_SEASON_ID', 'TEAM_GAME_DATE', 'TEAM_MATCHUP', 'TEAM_WL', 'VIDEO_AVAILABLE' ], inplace=True)
df.head()

,Unnamed: 0,PLAYER_NAME,PLAYER_ID,MATCHUP,TEAM_ABBREVIATION,TEAM_ID,OPP_ABBREVIATION,HOME_GAME,GAME_ID,GAME_DATE,WL,PTS,AST,REB,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,STL,BLK,TOV,PLUS_MINUS,FANTASY_PTS,POINT_PER_SHOT,EFG,START_POSITION,COMMENT,E_OFF_RATING,E_DEF_RATING,NET_RATING,OREB_PCT,DREB_PCT,REB_PCT,AST_PCT,EFG_PCT,AST_TOV,USG_PCT,TS_PCT,E_PACE,PACE,PIE,POSS,PACE_PER40,E_USG_PCT,MIN,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,PTS_OFF_TOV,PTS_2ND_CHANCE,PTS_FB,PTS_PAINT,OPP_PTS_OFF_TOV,OPP_PTS_2ND_CHANCE,OPP_PTS_FB,OPP_PTS_PAINT,BLKA,PF,PFD,IS_PLAYOFF,TEAM_SEASON_ID_x,whos_favored,spread,total,team_is_favored,team_spread,OPP_BLOWOUT_RISK,TEAM_SEASON_ID,TEAM_NAME,TEAM_GAME_DATE,TEAM_MATCHUP,TEAM_WL,TEAM_MIN,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_STL,TEAM_BLK,TEAM_TOV,TEAM_PF,TEAM_PTS,TEAM_PLUS_MINUS,VIDEO_AVAILABLE,TEAM_PACE,GAME_PACE,OPP_PACE,OPP_TEAM_ID,TEAM_OFF_RATING,TEAM_DEF_RATING,OPP_DEF_RATING,OPP_OFF_RATING,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV,percentageFieldGoalsAttempted2pt,percentageFieldGoalsAttempted3pt,percentagePoints2pt,percentagePointsMidrange2pt,percentagePoints3pt,percentagePointsFastBreak,percentagePointsFreeThrow,percentagePointsOffTurnovers,percentagePointsPaint,percentageAssisted2pt,percentageUnassisted2pt,percentageAssisted3pt,percentageUnassisted3pt,percentageAssistedFGM,percentageUnassistedFGM
0,0,Mike Conley,201144,MIN @ LAL,MIN,1610612750,LAL,0,22400062,2024-10-22,L,5,2,4,1,7,0.143,0,5,0.000,3,3,1.0,2,2,1,0,3,-22,12.8,0.601,0.142857,G,NaN,72.8,130.4,-57.0,0.091,0.091,0.091,0.222,0.143,0.67,0.220,0.300,101.33,96.16,0.016,41,80.13,0.229,20.22,4.45,1.62,3,3,6,55,0,0,40,0,2,0.00,1,5,0.200,1,1,1.000,0.0,5.0,0.0,2.0,17.0,8.0,15.0,38.0,1.0,1.0,1.0,0,22024,0,1.5,223.5,1,-1.5,0,22024,Minnesota Timberwolves,2024-10-22,MIN @ LAL,L,240,35,85,0.412,13,41,0.317,20,27,0.741,12,35,47,17,4,1,16,22,103,-7,1,99.4,99.4,99.4,1610612747,102.1,109.0,105.1,112.2,110.0,42.0,95.0,0.442,46.0,22.0,7.0,8.0,7.0,0.286,0.714,0.400,0.0,0.000,0.00,0.600,0.000,0.400,0.000,1.000,0.0,0.0,0.000,1.000
1,1,Dalton Knecht,1642261,LAL vs. MIN,LAL,1610612747,MIN,1,22400062,2024-10-22,W,5,2,1,2,4,0.500,1,3,0.333,0,0,NaN,0,1,1,0,1,7,11.2,1.250,0.625000,NaN,NaN,128.7,112.3,13.8,0.000,0.063,0.031,0.118,0.625,2.00,0.122,0.625,102.12,100.36,0.070,34,83.63,0.128,15.78,4.46,1.25,0,1,1,18,0,0,12,0,0,0.00,2,4,0.500,0,0,0.000,0.0,0.0,2.0,2.0,2.0,11.0,3.0,16.0,0.0,1.0,0.0,0,22024,0,1.5,223.5,0,1.5,0,22024,Los Angeles Lakers,2024-10-22,LAL vs. MIN,W,240,42,95,0.442,5,30,0.167,21,25,0.840,15,31,46,22,7,8,7,22,110,7,1,99.4,99.4,99.4,1610612750,112.2,105.1,109.0,102.1,103.0,35.0,85.0,0.412,47.0,17.0,4.0,1.0,16.0,0.250,0.750,0.400,0.0,0.600,0.40,0.000,0.000,0.400,0.000,1.000,1.0,0.0,0.500,0.500
2,2,Jaden McDaniels,1630183,MIN @ LAL,MIN,1610612750,LAL,0,22400062,2024-10-22,L,6,1,2,3,8,0.375,0,3,0.000,0,0,NaN,0,2,1,0,1,-8,11.9,0.750,0.375000,F,NaN,80.0,116.4,-26.7,0.000,0.105,0.059,0.143,0.375,1.00,0.243,0.375,95.76,90.00,-0.022,30,75.00,0.245,16.00,4.53,1.28,3,5,8,21,0,0,10,1,2,0.50,2,6,0.333,1,1,1.000,2.0,0.0,0.0,6.0,9.0,7.0,7.0,18.0,1.0,5.0,2.0,0,22024,0,1.5,223.5,1,-1.5,0,22024,Minnesota Timberwolves,2024-10-22,MIN @ LAL,L,240,35,85,0.412,13,41,0.317,20,27,0.741,12,35,47,17,4,1,16,22,103,-7,1,99.4,99.4,99.4,1610612747,102.1,109.0,105.1,112.2,110.0,42.0,95.0,0.442,46.0,22.0,7.0,8.0,7.0,0.625,0.375,1.000,0.0,0.000,0.00,0.000,0.333,1.000,0.667,0.333,0.0,0.0,0.667,0.333
3,3,Naz Reid,1629675,MIN @ LAL,MIN,1610612750,LAL,0,22400062,2024-10-22,L,12,1,4,3,8,0.375,2,4,0.500,4,4,1.0,1,3,0,0,1,-6,17.3,1.230,0.500000,NaN,NaN,112.9,118.4,-9.0,0.034,0.107,0.070,0.056,0.500,1.00,0.172,0.615,100.77,97.46,0.074,53,81.21,0.174,26.35,4.22,2.00,3,9,12,34,0,0,23,1,4,0.25,2,4,0.500,5,8,0.625,2.0,2.0,3.0,2.0,8.0,13.0,8.0,46.0,0.0,3.0,2.0,0,22024,0,1.5,223.5,1,-1.5,0,2202

## Fetches Player Gamelogs

In [2]:
# nba = FetchPlayersStats()
# data = nba.getCompleteStats(
#     season='2018-19', 
#     season_type='Regular Season', 
#     sleep_time=1.5, 
#     max_workers=5,
#     batch_limit=100,
#     complete_cache_file='../DATA/CSV_FILES/REGULAR_DATA/S19.csv'
# )
# data.head()

## Merge team data into player logs

In [4]:
# # Drop all team-related columns to re-merge with correct pace calculations
# data = s19_regular

# columns_to_drop = [
#     'Unnamed: 0.3', 'Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0',
#     # Pace columns (incorrect calculations)
#     'TEAM_PACE', 'GAME_PACE', 'OPP_PACE',
    
#     # Rating columns (may need recalculation with correct pace)
#     'TEAM_OFF_RATING', 'TEAM_DEF_RATING',
#     'OPP_OFF_RATING', 'OPP_DEF_RATING',
    
#     # Other team stats that came from the merge
#     'TEAM_PTS', 'TEAM_FGM', 'TEAM_FGA', 'TEAM_FG_PCT',
#     'TEAM_FG3M', 'TEAM_FG3A', 'TEAM_FG3_PCT',
#     'TEAM_FTM', 'TEAM_FTA', 'TEAM_FT_PCT', 
#     'TEAM_OREB', 'TEAM_DREB', 'TEAM_REB',
#     'TEAM_AST', 'TEAM_STL', 'TEAM_BLK', 'TEAM_TOV',
#     'TEAM_PF', 'TEAM_PLUS_MINUS',
    
#     # Opponent stats
#     'OPP_TEAM_ID', 'OPP_PTS', 'OPP_FGM', 'OPP_FGA', 'OPP_FG_PCT',
#     'OPP_REB', 'OPP_AST', 'OPP_STL', 'OPP_BLK', 'OPP_TOV'
    
#     'TEAM_SEASON_ID_x',	'OPP_TOV_x', 'TEAM_SEASON_ID_y', 'OPP_TOV_y', 'TEAM_SEASON_ID', 'TEAM_NAME', 'TEAM_GAME_DATE', 'TEAM_MATCHUP', 'TEAM_WL', 'TEAM_MIN', 'VIDEO_AVAILABLE'
# ]

# # Drop columns that exist in the dataframe
# existing_cols = [col for col in columns_to_drop if col in data.columns]
# data = data.drop(columns=existing_cols)

# teamlogs = mergeTeamtoPlayer(data, season='2018-19', season_type='Regular Season')
# teamlogs


In [5]:
# teamlogs.to_csv('../DATA/CSV_FILES/REGULAR_DATA/S19.csv')
# teamlogs

## Assign features for regular season data

In [4]:
star_players_by_year = {
    2019:[ "Giannis Antetokounmpo",
    "LeBron James",
    "Anthony Davis",
    "James Harden",
    "Luka Dončić",
    "Kawhi Leonard",
    "Pascal Siakam",
    "Nikola Jokić",
    "Damian Lillard",
    "Chris Paul",
    "Jayson Tatum",
    "Jimmy Butler",
    "Rudy Gobert",
    "Ben Simmons",
    "Russell Westbrook"],
    2020:[
    "Giannis Antetokounmpo",
    "Kawhi Leonard", 
    "Nikola Jokić",
    "Stephen Curry",
    "Luka Dončić",
    "Julius Randle",
    "LeBron James",
    "Joel Embiid",
    "Chris Paul",
    "Damian Lillard",
    "Jimmy Butler",
    "Paul George",
    "Rudy Gobert",
    "Bradley Beal",
    "Kyrie Irving"
    ],
    2021: [
        "Giannis Antetokounmpo", "Kawhi Leonard", "Nikola Jokić", "Stephen Curry", "Luka Dončić",
        "Julius Randle", "LeBron James", "Joel Embiid", "Damian Lillard", "Chris Paul",
        "Jimmy Butler", "Paul George", "Rudy Gobert", "Bradley Beal", "Kyrie Irving",
        "Devin Booker", "Mike Conley", "James Harden", "Zach LaVine", "Donovan Mitchell",
        "Nikola Vucevic", "Anthony Davis"
    ],
    2022: [
        "Giannis Antetokounmpo", "Luka Dončić", "Jayson Tatum", "Nikola Jokić", "Devin Booker",
        "Ja Morant", "Stephen Curry", "DeMar DeRozan", "Kevin Durant", "Joel Embiid",
        "LeBron James", "Chris Paul", "Trae Young", "Pascal Siakam", "Karl-Anthony Towns",
        "Andrew Wiggins", "Donovan Mitchell", "Rudy Gobert", "Zach LaVine", "Khris Middleton",
        "Jimmy Butler", "Darius Garland", "Fred VanVleet", "LaMelo Ball"
    ],
    2023: [
        "Giannis Antetokounmpo", "Jayson Tatum", "Joel Embiid", "Shai Gilgeous-Alexander", "Luka Dončić",
        "Jaylen Brown", "Jimmy Butler", "Nikola Jokić", "Stephen Curry", "Donovan Mitchell",
        "LeBron James", "Julius Randle", "Domantas Sabonis", "De'Aaron Fox", "Damian Lillard",
        "Kyrie Irving", "Zion Williamson", "Kevin Durant", "Ja Morant", "DeMar DeRozan",
        "Tyrese Haliburton", "Jrue Holiday", "Bam Adebayo", "Jaren Jackson Jr.", "Paul George",
        "Pascal Siakam", "Anthony Edwards"
    ],
    2024: [
        "Shai Gilgeous-Alexander", "Luka Dončić", "Jayson Tatum", "Giannis Antetokounmpo", "Nikola Jokić",
        "Jalen Brunson", "Anthony Edwards", "Kawhi Leonard", "Kevin Durant", "Anthony Davis",
        "Stephen Curry", "Devin Booker", "LeBron James", "Domantas Sabonis", "Bam Adebayo",
        "Tyrese Haliburton", "Damian Lillard", "Karl-Anthony Towns", "Jaylen Brown",
        "Trae Young", "Paolo Banchero", "Scottie Barnes", 'Joel Embiid'
    ],
    2025: [
        'Shai Gilgeous-Alexander', 'Nikola Jokić', 'Giannis Antetokounmpo', 'Jayson Tatum', 'Donovan Mitchell',
        'Anthony Edwards', 'LeBron James', 'Stephen Curry', 'Evan Mobley', 'Jalen Brunson',
        'Cade Cunningham', 'Karl-Anthony Towns', 'Tyrese Haliburton', 'Jalen Williams', 'James Harden',
        'Darius Garland', 'Damian Lillard', 'Anthony Davis', 'Kyrie Irving', 'Jaylen Brown', 'Tyler Herro', 'Jaren Jackson Jr.', 
        'Pascal Siakam', 'Victor Wembanyama', 'Alperen Sengun', 'Trae Young', 'LaMelo Ball', 'Devin Booker', 'Joel Embiid', 'Luka Dončić'
]}

In [5]:
def process_season_features(season_df, prop_type, year, star_players):
    df = season_df.copy()
    df = sort_data_for_features(df)
    df['STARTING'] = df['START_POSITION'].apply(lambda x: 1 if x in ['G','F','C'] else 0)
    # df['BLOWOUT_RISK'] = (abs(df['spread']) >= 10).astype(int)
    df['TEAM_IMPLIED_PTS_FAV'] = ((df['total'] + df['team_spread']) / 2).round(1)
    df['TEAM_IMPLIED_PTS_UND'] = ((df['total'] - df['team_spread']) / 2).round(1)
    
    # Add position data
    cache_file = os.path.join(feature_path, 'DATA', 'TOOLS', 'playerInfo.csv')
    df = assign_position_with_cache(
        df, 
        cache_file=cache_file,
        max_workers=4, 
        delay_between_requests=1.5
    )
    # Basic features to track rest and travel
    df = add_rest_day_features(df)
    
    # Prop-specific features
    df = assign_team_opp_def_by_position(df)
    df = teamRollingDefenseByPosition(df, team_id_col='TEAM_ID', date_col='GAME_DATE', windows=[5,10,15])
    df = assign_team_opp_zone_by_position(df)
    df = rollingAverages(df, player_id_col='PLAYER_ID', date_col='GAME_DATE', windows=[5,10,15,40])
    df = statAgainstTeam(df, player_id_col='PLAYER_ID', opp_col='OPP_ABBREVIATION', stat_line=prop_type)
    df = HomeAwayAverages(df, player_id_col='PLAYER_ID', date_col='GAME_DATE')
    df = getPlayerAvgToDateVectorized(df)
    df = addLagFeatures(df, stat_lines=['PTS', 'MIN', 'FGA', 'FTA', 'FG3A', 'USG_PCT', 'TS_PCT'])
    # df = add_volatility_features(df, windows=[3,5,7,15,40])
    # df = add_performance_volatility_categories(df)
    # df = add_recent_form_volatility(df)
    df = process_star_players_data(df, star_players, min_minutes=20)
    df = add_performance_without_stars_columns(df, min_games=1)
    df = teamUsualStarters(df)
    df = oppTeamUsualStarters(df)
    # df = add_all_defensive_features(df, all_defensive_players, year)
    df = teamContext(df)
    df = assign_opponent_team_stats_dict(df)
    df = expectedPace(df)
    
    # Clean up any unwanted columns
    if 'Unnamed: 0' in df.columns:
        df.drop(columns=['Unnamed: 0'], inplace=True)
    if 'Unnamed: 0.1' in df.columns:
        df.drop(columns=['Unnamed: 0.1'], inplace=True)
    return df

prop_types = ['PTS']
data = [s19_regular,s20_regular, s21_regular,s22_regular,s23_regular,s24_regular,s25_regular]
seasons = [2019,2020,2021,2022,2023,2024,2025]

# Pre-define output directory once
output_dir = os.path.join(feature_path, 'DATA', 'CSV_FILES', 'TRAIN_DATA')
os.makedirs(output_dir, exist_ok=True)

# Process each season sequentially but with optimized operations
for season_data, year in zip(data, seasons):
    print(f"Processing year {year}...")
    
    # Process features
    processed_data = process_season_features(
        season_data, 
        prop_type='PTS',
        year=year,
        star_players=set(star_players_by_year[year]),
        # all_defensive_players=set(all_defensive_players[year])
    )
    
    # Save file
    output_path = os.path.join(output_dir, f'PTS_TRAIN_{str(year)[-2:]}.csv')
    processed_data.to_csv(output_path)
    print(f"Completed {year}")

Processing year 2019...
Loading position cache...
Loaded 1175 players from cache
Found 530 unique players, 0 need to be fetched
Saving updated cache...
Cache saved with 1175 players
Applying positions to dataset...
Creating position flags...
Position assignment completed!


/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = df.groupby(player_id_col)[col].transform(
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = df.groupby(player_id_col)[col].transform(
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:124: PerformanceWarning: D

Completed 2019
Processing year 2020...
Loading position cache...
Loaded 1175 players from cache
Found 529 unique players, 0 need to be fetched
Saving updated cache...
Cache saved with 1175 players
Applying positions to dataset...
Creating position flags...
Position assignment completed!


/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = df.groupby(player_id_col)[col].transform(
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = df.groupby(player_id_col)[col].transform(
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:124: PerformanceWarning: D

Completed 2020
Processing year 2021...
Loading position cache...
Loaded 1175 players from cache
Found 540 unique players, 0 need to be fetched
Saving updated cache...
Cache saved with 1175 players
Applying positions to dataset...
Creating position flags...
Position assignment completed!


...

Completed 2023
Processing year 2024...
Loading position cache...
Loaded 1175 players from cache
Found 572 unique players, 0 need to be fetched
Saving updated cache...
Cache saved with 1175 players
Applying positions to dataset...
Creating position flags...
Position assignment completed!


/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = df.groupby(player_id_col)[col].transform(
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = df.groupby(player_id_col)[col].transform(
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:124: PerformanceWarning: D

Completed 2024
Processing year 2025...
Loading position cache...
Loaded 1175 players from cache
Found 569 unique players, 0 need to be fetched
Saving updated cache...
Cache saved with 1175 players
Applying positions to dataset...
Creating position flags...
Position assignment completed!


/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = df.groupby(player_id_col)[col].transform(
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[rolling_col_name] = df.groupby(player_id_col)[col].transform(
/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/FEATURE_ENGINEERING/features.py:124: PerformanceWarning: D

Completed 2025


In [8]:
# df = s21

# df['STARTING'] = df['START_POSITION'].apply(lambda x: 1 if x in ['G','F','C'] else 0)
# df['team_is_favored'] = df['team_is_favored'].astype(int)
# df['whos_favored'] = df['whos_favored'].apply(lambda x: 1 if x == 'home' else 0)
# df['BLOWOUT_RISK'] = (abs(df['spread']) >= 10).astype(int)

# # Add position data
# cache_file = os.path.join(feature_path, 'DATA', 'TOOLS', 'playerInfo.csv')
# df = assign_position_with_cache(
#     df, 
#     cache_file=cache_file,
#     max_workers=4, 
#     delay_between_requests=1.5
# )
# df = add_rest_day_features(df)
# df = add_minutes_trend_features(df)
# df = add_lineup_cohesion(df)
# df = add_rotation_stability(df)
# df = add_usage_shift(df)
# df = MINrollingAverages(df, player_id_col='PLAYER_ID', date_col='GAME_DATE', windows=[3,5,7])
# df = MINLagFeatures(df, player_id_col='PLAYER_ID', date_col='GAME_DATE', stat_line='MIN')
# df = getPlayerMINAvgToDate(df, player_id_col='PLAYER_ID', date_col='GAME_DATE')
# df = MINHomeAwayAverages(df, player_id_col='PLAYER_ID', date_col='GAME_DATE')
# df = MINAgainstTeam(df, player_id_col='PLAYER_ID', opp_col='OPP_ABBREVIATION', stat_line='MIN')
# df = assign_opponent_team_stats_dict(df)
# df = teamContext(df)
# df.to_csv('../DATA/CSV_FILES/TRAIN_DATA/MIN_TRAIN_21.csv', index=False)

In [9]:
def merge_betting_data(player_df, betting_df, team_dict):
    """
    Merge betting data (spread, total, who's favored) into player dataset
    """
    df = player_df.copy()
    odds = betting_df.copy()
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    odds['date'] = pd.to_datetime(odds['date'])
    
    # Convert betting data team abbreviations to uppercase using team_dict
    odds['away_upper'] = odds['away'].map(team_dict)
    odds['home_upper'] = odds['home'].map(team_dict)
    
    # First, create a unique identifier for each game in odds data
    odds['game_key_home'] = odds['date'].astype(str) + '_' + odds['home_upper'] + '_' + odds['away_upper']
    odds['game_key_away'] = odds['date'].astype(str) + '_' + odds['away_upper'] + '_' + odds['home_upper']
    
    df['game_key'] = df['GAME_DATE'].astype(str) + '_' + df['TEAM_ABBREVIATION'] + '_' + df['OPP_ABBREVIATION']
    home_merge = df.merge(
        odds[['game_key_home', 'whos_favored', 'spread', 'total']].rename(columns={'game_key_home': 'game_key'}),
        on='game_key',
        how='left',
        suffixes=('', '_home')
    )
    away_merge = df.merge(
        odds[['game_key_away', 'whos_favored', 'spread', 'total']].rename(columns={'game_key_away': 'game_key'}),
        on='game_key', 
        how='left',
        suffixes=('', '_away')
    )
    df['whos_favored'] = home_merge['whos_favored'].fillna(away_merge['whos_favored'])
    df['spread'] = home_merge['spread'].fillna(away_merge['spread']).round(2)
    df['total'] = home_merge['total'].fillna(away_merge['total']).round(2)
    df['team_is_favored'] = ((df['whos_favored'] == 'home') & (df['HOME_GAME'] == 1)) | \
                           ((df['whos_favored'] == 'away') & (df['HOME_GAME'] == 0))
    df['team_spread'] = df.apply(lambda row: 
        round(row['spread'] if row['HOME_GAME'] == 1 else -row['spread'], 2), axis=1)
    df.drop('game_key', axis=1, inplace=True)
    return df

team_dict = {
    'min': 'MIN', 
    'bos': 'BOS', 
    'bkn': 'BKN', 
    'ny': 'NYK', 
    'phi': 'PHI', 
    'tor': 'TOR', 
    'chi': 'CHI', 
    'cle': 'CLE', 
    'det': 'DET', 
    'ind': 'IND', 
    'mia': 'MIA', 
    'atl': 'ATL', 
    'cha': 'CHA', 
    'was': 'WAS',
    'wsh': 'WAS',
    'orl': 'ORL', 
    'mil': 'MIL', 
    'chh': 'CHH', 
    'dal': 'DAL', 
    'hou': 'HOU',
    'lac': 'LAC',
    'lal': 'LAL',
    'sac': 'SAC',
    'por': 'POR',
    'uta': 'UTA',
    'utah': 'UTA', 
    'den': 'DEN',
    'okc': 'OKC',
    'mem': 'MEM',
    'no': 'NOP',
    'sa': 'SAS',    
    'gs': 'GSW',
    'phx': 'PHX',  
}

In [10]:
bettingData = pd.read_csv('../DATA/CSV_FILES/bettingData.csv')
df = merge_betting_data(s20_regular, bettingData, team_dict)
df

,Unnamed: 0.1,Unnamed: 0,PLAYER_NAME,PLAYER_ID,MATCHUP,TEAM_ABBREVIATION,TEAM_ID,OPP_ABBREVIATION,HOME_GAME,GAME_ID,GAME_DATE,WL,PTS,AST,REB,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,STL,BLK,TOV,PLUS_MINUS,FANTASY_PTS,POINT_PER_SHOT,EFG,START_POSITION,COMMENT,OFF_RATING,E_OFF_RATING,DEF_RATING,E_DEF_RATING,NET_RATING,OREB_PCT,DREB_PCT,REB_PCT,AST_PCT,EFG_PCT,AST_TOV,USG_PCT,TS_PCT,E_PACE,PACE,PIE,POSS,PACE_PER40,E_USG_PCT,MIN,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,PTS_OFF_TOV,PTS_2ND_CHANCE,PTS_FB,PTS_PAINT,OPP_PTS_OFF_TOV,OPP_PTS_2ND_CHANCE,OPP_PTS_FB,OPP_PTS_PAINT,BLKA,PF,PFD,IS_PLAYOFF,TEAM_SEASON_ID,TEAM_NAME,TEAM_GAME_DATE,TEAM_MATCHUP,TEAM_WL,TEAM_MIN,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_STL,TEAM_BLK,TEAM_TOV,TEAM_PF,TEAM_PTS,TEAM_PLUS_MINUS,VIDEO_AVAILABLE,TEAM_PACE,GAME_PACE,OPP_PACE,OPP_TEAM_ID,TEAM_OFF_RATING,TEAM_DEF_RATING,OPP_DEF_RATING,OPP_OFF_RATING,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV,whos_favored,spread,total,team_is_favored,team_spread
0,0,0,Derrick Favors,202324,NOP @ TOR,NOP,1610612740,TOR,0,21900001,2019-10-22,L,6,2,7,3,6,0.500,0,0,NaN,0,0,NaN,1,6,0,1,1,-12,19.4,1.000,0.500000,C,NaN,108.2,108.2,132.7,132.7,-24.5,0.048,0.261,0.159,0.118,0.500,2.0,0.137,0.500,113.31,113.31,0.056,49,94.43,0.137,20.75,4.60,1.75,4,8,10,43,0,0,35,3,3,1.000,0,3,0.000,7,8,0.875,0.0,2.0,0.0,6.0,12.0,11.0,7.0,32.0,0.0,5.0,0.0,0,22019,New Orleans Pelicans,2019-10-22,NOP @ TOR,L,265,43,102,0.422,19,45,0.422,17,20,0.850,16,37,53,30,4,9,19,34,122,-8,1,106.2,106.2,106.2,1610612761,107.2,114.2,101.1,107.7,130.0,42.0,103.0,0.408,57.0,23.0,7.0,3.0,17.0,home,6.5,229.5,False,-6.5
1,1,1,Brandon Ingram,1627742,NOP @ TOR,NOP,1610612740,TOR,0,21900001,2019-10-22,L,22,5,5,8,19,0.421,2,5,0.400,4,4,1.0,0,5,1,2,2,-19,42.5,1.060,0.473684,F,NaN,102.6,102.6,122.5,122.5,-19.9,0.000,0.125,0.068,0.227,0.474,2.5,0.272,0.530,107.35,107.35,0.112,77,89.46,0.272,35.10,4.33,2.78,1,9,9,69,2,0,46,5,9,0.556,3,10,0.300,4,8,0.500,3.0,0.0,3.0,8.0,16.0,14.0,14.0,44.0,0.0,4.0,6.0,0,22019,New Orleans Pelicans,2019-10-22,NOP @ TOR,L,265,43,102,0.422,19,45,0.422,17,20,0.850,16,37,53,30,4,9,19,34,122,-8,1,106.2,106.2,106.2,1610612761,107.2,114.2,101.1,107.7,130.0,42.0,103.0,0.408,57.0,23.0,7.0,3.0,17.0,home,6.5,229.5,False,-6.5
2,2,2,Josh Hart,1628404,NOP @ TOR,NOP,1610612740,TOR,0,21900001,2019-10-22,L,15,1,10,4,9,0.444,3,5,0.600,4,4,1.0,4,6,0,1,1,-1,30.5,1.394,0.611111,NaN,NaN,105.5,105.5,101.7,101.7,3.7,0.111,0.167,0.139,0.067,0.611,1.0,0.174,0.697,96.28,96.28,0.213,55,80.24,0.174,28.17,4.31,2.22,5,8,13,42,0,0,27,2,5,0.400,2,4,0.500,2,4,0.500,0.0,4.0,2.0,2.0,10.0,5.0,12.0,22.0,1.0,4.0,4.0,0,22019,New Orleans Pelicans,2019-10-22,NOP @ TOR,L,265,43,102,0.422,19,45,0.422,17,20,0.850,16,37,53,30,4,9,19,34,122,-8,1,106.2,106.2,106.2,1610612761,107.2,114.2,101.1,107.7,130.0,42.0,103.0,0.408,57.0,23.0,7.0,3.0,17.0,home,6.5,229.5,False,-6.5
3,3,3,Lou Williams,101150,LAC vs. LAL,LAC,1610612746,LAL,1,21900002,2019-10-22,W,21,7,5,8,14,0.571,1,4,0.250,4,4,1.0,1,4,1,0,2,13,38.5,1.332,0.607143,NaN,NaN,121.6,121.6,102.7,102.7,19.0,0.029,0.098,0.066,0.280,0.607,3.5,0.214,0.666,97.38,97.38,0.195,74,81.15,0.214,36.72,3.76,2.46,1,5,6,89,0,2,62,1,2,0.500,7,12,0.583,0,1,0.000,8.0,2.0,5.0,6.0,9.0,6.0,5.0,30.0,1.0,0.0,9.0,0,22019,LA Clippers,2019-10-22,LAC vs. LAL,W,240,42,81,0.519,11,31,0.355,17,24,0.708,11,34,45,24,8,5,14,25,112,10,1,97.4,97.4,97.4,1610612747,118.4,107.9,111.7,101.8,102.0,37.0,85.0,0.435,41.0,20.0,4.0,7.0,15.0,away,3.5,224.0,False,3.5
4,4,4,Patrick Beverley,201976,LAC vs. LAL,LAC,1610612746,LAL,1,21900002,2019-10-22,W,2,6,10,1,7,0.143,0,5,0.000,0,0,NaN,2,8,0,1,2,13,24.0,0.286,0.142857,G,NaN,120.0,120.0,100.0,100.0,20.0,0.074,0.276,0.179,0.207,0.143,3.0,0.122,0.143,99.51,99.51,0.048,65,82.93,0.122,31.35,4.24,2.37,4,11,13,62,2,0,51,1,2,0.500,0,5,0.000,1,2,0.500,0.0,0.0